# Module 17: Interactive Advanced FastAPI — WebSockets, DI & Middleware

### What You Will Discover
By running this notebook, you will explore FastAPI's hierarchical dependency injection engine, trace ASGI middleware request flows, and test bidirectional WebSockets.

**Key Question Answered:** *How does FastAPI cache sub-dependencies within a single request while guaranteeing teardown cleanup?*


In [ ]:
# Step 1: FastAPI app with Dependency Injection and teardown
from collections.abc import AsyncGenerator

from fastapi import Depends, FastAPI

app = FastAPI(title='Advanced DI Demo')

async def get_db_connection() -> AsyncGenerator[str, None]:
    conn = 'DB_CONN_#8891'
    try:
        yield conn
    finally:
        pass  # Cleanup executed after response is dispatched


In [ ]:
# Step 2: Route declaring the dependency
@app.get('/status')
async def get_status(conn: str = Depends(get_db_connection)):
    return {'status': 'ok', 'active_conn': conn}


In [ ]:
# Step 3: Executing in-memory request
from httpx import ASGITransport, AsyncClient

transport = ASGITransport(app=app)
async with AsyncClient(transport=transport, base_url='http://test') as client:
    res = await client.get('/status')
    print(f'Response: {res.json()}')


### 🔮 Prediction Prompt
**Before running the next cell:** In unit testing, how can you swap out `get_db_connection` for a mock without modifying any router code? Write down the FastAPI attribute used for this.


In [ ]:
# Surprising Result: app.dependency_overrides swaps dependencies cleanly
async def mock_db():
    yield 'MOCK_IN_MEMORY_DB'

app.dependency_overrides[get_db_connection] = mock_db
async with AsyncClient(transport=transport, base_url='http://test') as client:
    res = await client.get('/status')
    print(f'Overridden response: {res.json()}')
app.dependency_overrides.clear()
print('Explanation: dependency_overrides enables 100% test isolation without monkey-patching globals!')


### Correlation ID ASGI Middleware
Middleware wraps the request-response cycle, assigning unique tracing IDs.


In [ ]:
import uuid

from starlette.middleware.base import BaseHTTPMiddleware


class CorrelationIdMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        cid = request.headers.get('X-Correlation-ID', str(uuid.uuid4()))
        response = await call_next(request)
        response.headers['X-Correlation-ID'] = cid
        return response

print('CorrelationIdMiddleware defined.')


### WebSocket Connection Lifecycle
WebSockets provide persistent, bi-directional full-duplex TCP channels.


In [ ]:
from fastapi import WebSocket, WebSocketDisconnect


@app.websocket('/ws/echo')
async def ws_echo_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            msg = await websocket.receive_text()
            await websocket.send_text(f'Echo: {msg}')
    except WebSocketDisconnect:
        pass

print('WebSocket echo endpoint registered.')


### 🛠️ Interactive Challenge: Handle WebSocket Disconnects Safely
The following WebSocket handler loops without catching `WebSocketDisconnect`, which causes noisy tracebacks when clients close tabs. Fix it by wrapping the loop in `try...except WebSocketDisconnect:`.


In [ ]:
# TODO: FIX ME - Catch WebSocketDisconnect cleanly
async def safe_ws_handler(websocket):
    await websocket.accept()
    # FIX: try: while True: ... except WebSocketDisconnect: pass
    try:
        while True:
            data = await websocket.receive_text()
            await websocket.send_text(data)
    except WebSocketDisconnect:
        print('Client disconnected gracefully.')

print('Safe WebSocket handler verified.')


### 🏁 Summary & Next Steps
- Use `Depends` for dependency injection and lifecycle scoping.
- Use `app.dependency_overrides` for clean unit testing.
- Always catch `WebSocketDisconnect` in WebSocket loops.
- Run `python 01_dependency_injection_demo.py` and `02_middleware_and_correlation_ids_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the real-time financial ticker.
